# Week 5 – Day 4
# CrewAI: Multi-Agent Collaboration, Roles & Task Delegation

# Task 1 – Multi-Agent Design Thinking

For this task, we design a small CrewAI team capable of completing a realistic remote sensing data analysis workflow.

### Selected Business Task

**Environmental Monitoring Report Generation**

An environmental consulting organization has received a CSV dataset exported from a GIS/remote sensing workflow. The dataset contains vegetation indices (NDVI), land cover information, rainfall measurements, and other environmental observations collected from multiple monitoring sites.

The objective is to transform the raw dataset into a stakeholder-ready environmental assessment report by performing the following steps:

1. Inspect and validate the dataset for missing or inconsistent observations.
2. Analyze the cleaned data to identify vegetation health patterns and environmental trends.
3. Produce a professional environmental monitoring report for decision-makers.

This workflow naturally separates into specialized responsibilities, making it an ideal candidate for a multi-agent system.

## Agent Design
<center>
<img src="environmental_monitoring_crew.png" width="900">
</center>
The environmental monitoring workflow is divided among three specialized agents. Each agent has a clearly defined responsibility with minimal overlap.

| Agent | Role | Primary Responsibility |
|--------|------|------------------------|
| Agent 1 | Environmental Data Quality Specialist | Validate and prepare the remote sensing dataset for analysis |
| Agent 2 | Remote Sensing Analyst | Analyze vegetation health, land cover, and environmental trends |
| Agent 3 | Environmental Assessment Report Writer | Produce a stakeholder-ready environmental report based on the analysis |

This separation follows the principle of specialization, allowing each agent to focus on a single well-defined task.


In [1]:
agent_design = {
    "Environmental Data Quality Specialist": {
        "Goal": (
            "Inspect, validate, and prepare environmental monitoring datasets "
            "for reliable analysis."
        ),
        "Backstory": (
            "An experienced environmental data specialist who identifies "
            "missing values, inconsistent records, abnormal sensor readings, "
            "and data quality issues before scientific analysis begins."
        )
    },

    "Remote Sensing Analyst": {
        "Goal": (
            "Analyze cleaned remote sensing data to identify vegetation "
            "health patterns and environmental trends."
        ),
        "Backstory": (
            "A remote sensing analyst experienced in interpreting NDVI, "
            "land cover classifications, rainfall observations, and "
            "environmental indicators to generate actionable insights."
        )
    },

    "Environmental Assessment Report Writer": {
        "Goal": (
            "Transform technical environmental analyses into clear, "
            "professional reports for decision-makers."
        ),
        "Backstory": (
            "An environmental reporting specialist who communicates complex "
            "scientific findings in a concise and accessible manner for "
            "government agencies, environmental consultants, and stakeholders."
        )
    }
}

for role, info in agent_design.items():
    print(f"\n{'='*70}")
    print(f"ROLE: {role}")
    print(f"Goal: {info['Goal']}")
    print(f"Backstory: {info['Backstory']}")


ROLE: Environmental Data Quality Specialist
Goal: Inspect, validate, and prepare environmental monitoring datasets for reliable analysis.
Backstory: An experienced environmental data specialist who identifies missing values, inconsistent records, abnormal sensor readings, and data quality issues before scientific analysis begins.

ROLE: Remote Sensing Analyst
Goal: Analyze cleaned remote sensing data to identify vegetation health patterns and environmental trends.
Backstory: A remote sensing analyst experienced in interpreting NDVI, land cover classifications, rainfall observations, and environmental indicators to generate actionable insights.

ROLE: Environmental Assessment Report Writer
Goal: Transform technical environmental analyses into clear, professional reports for decision-makers.
Backstory: An environmental reporting specialist who communicates complex scientific findings in a concise and accessible manner for government agencies, environmental consultants, and stakeholders.

## Why Multiple Specialized Agents?

Environmental data analysis involves multiple stages that require different areas of expertise, from validating raw observations to interpreting environmental indicators and communicating findings to stakeholders. Assigning these responsibilities to specialized agents improves task focus, produces more structured intermediate outputs, and enables each stage of the workflow to build on the previous one.

However, for small datasets or straightforward reporting tasks, a single well-designed agent may be sufficient. In such cases, the additional coordination overhead of a multi-agent system may not justify the increased complexity or token usage.

# Task 2 – Build Agents & Assign Tools

In this task, we implement the three specialized CrewAI agents designed in Task 1.

Each agent receives:
- its own LLM configuration,
- only the tools required for its responsibility,
- a role, goal, and backstory that reflect its specialization.

Following the principle of least privilege, each agent is provided only with the tools necessary for its assigned responsibility. This encourages specialization, reduces unnecessary tool usage, and creates a more realistic multi-agent workflow.

In [2]:
# Imports
from crewai import Agent, LLM
from crewai.tools import tool

from dotenv import load_dotenv
import os
import pandas as pd

In [33]:
# Environment and LLM
load_dotenv()

API_KEY = os.getenv("NETIXSOL_API_KEY")
BASE_URL = "https://llm.netixsol.com/v1"

# CrewAI / LiteLLM compatibility
os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_BASE_URL"] = BASE_URL

MODEL = "batch"

llm = LLM(
    model=f"openai/{MODEL}",
    api_key=API_KEY,
    base_url="https://llm.netixsol.com/v1",
    temperature=0
)

## Tool Design

The environmental monitoring workflow requires different capabilities at different stages.

Rather than giving every agent access to every available tool, each agent receives only the tools needed to perform its specific responsibility.

This mirrors how real environmental and GIS teams distribute responsibilities among specialists.

In [4]:
# Tools

#tool # 1
@tool
def dataset_quality_report(file_path: str) -> str:
    """
    Generate a data quality report for an environmental monitoring dataset.
    """
    df = pd.read_csv(file_path)

    report = []

    report.append("=== DATASET OVERVIEW ===")
    report.append(f"Rows: {df.shape[0]}")
    report.append(f"Columns: {df.shape[1]}")
    report.append(f"\nColumns: {list(df.columns)}")

    report.append("\n=== MISSING VALUES ===")
    report.append(df.isnull().sum().to_string())

    report.append(f"\nDuplicate Rows: {df.duplicated().sum()}")

    # NDVI validation
    if "NDVI" in df.columns:
        invalid_ndvi = df[(df["NDVI"] < -1) | (df["NDVI"] > 1)]
        report.append(f"\nInvalid NDVI Values: {len(invalid_ndvi)}")

    # Rainfall validation
    if "Rainfall_mm" in df.columns:
        negative_rainfall = df[df["Rainfall_mm"] < 0]
        report.append(f"Negative Rainfall Values: {len(negative_rainfall)}")

    return "\n".join(report)

In [5]:
# tool # 2
@tool
def environmental_statistics(file_path: str) -> str:
    """
    Generate descriptive statistics for an environmental monitoring dataset.
    """
    df = pd.read_csv(file_path)

    report = []

    report.append("=== NUMERICAL SUMMARY ===")
    report.append(df.describe().to_string())

    if "Land_Cover" in df.columns:
        report.append("\n\n=== LAND COVER DISTRIBUTION ===")
        report.append(df["Land_Cover"].value_counts().to_string())

    if "Vegetation_Health" in df.columns:
        report.append("\n\n=== VEGETATION HEALTH ===")
        report.append(df["Vegetation_Health"].value_counts().to_string())

    return "\n".join(report)

In [6]:
# Agent 1

environmental_data_quality_specialist = Agent(
    role="Environmental Data Quality Specialist",

    goal=(
        "Inspect, validate, and prepare environmental monitoring datasets "
        "for reliable analysis."
    ),

    backstory=(
        "You are an experienced environmental data specialist responsible "
        "for identifying missing values, inconsistent records, abnormal "
        "sensor observations, and other data quality issues before analysis."
    ),

    llm=llm,

    tools=[dataset_quality_report],

    verbose=True
)

In [7]:
# Agent 2
remote_sensing_analyst = Agent(
    role="Remote Sensing Analyst",

    goal=(
        "Analyze cleaned environmental datasets to identify vegetation "
        "health patterns, land cover trends, and environmental insights."
    ),

    backstory=(
        "You are an experienced remote sensing analyst who specializes in "
        "interpreting NDVI values, rainfall observations, land cover "
        "information, and other environmental indicators."
    ),

    llm=llm,

    tools=[environmental_statistics],

    verbose=True
)

In [8]:
# Agent 3

environmental_report_writer = Agent(
    role="Environmental Assessment Report Writer",

    goal=(
        "Produce clear, professional environmental assessment reports "
        "for decision-makers and stakeholders."
    ),

    backstory=(
        "You are an environmental reporting specialist who transforms "
        "technical environmental analyses into concise, well-structured "
        "reports suitable for policymakers, researchers, and environmental "
        "consultants."
    ),

    llm=llm,

    tools=[],

    verbose=True
)

## Tool Assignment Justification

| Agent | Assigned Tool(s) | Justification |
|--------|------------------|---------------|
| Environmental Data Quality Specialist | `dataset_quality_report` | Generates a comprehensive quality assessment, including missing values, duplicate records, and invalid environmental measurements before analysis begins. |
| Remote Sensing Analyst | `environmental_statistics` | Produces descriptive statistics and summaries of land cover and vegetation health, enabling the analyst to identify environmental patterns and trends. |
| Environmental Assessment Report Writer | No tools | Focuses on communicating the analyst's findings in a clear, stakeholder-friendly report without directly processing the dataset. |

Assigning only role-appropriate tools encourages specialization, minimizes unnecessary tool usage, and reflects how responsibilities are distributed among environmental data professionals in real-world projects.

# Task 3 – Define Tasks & Process

In this task, we define the workflow that connects our three specialized agents.

Each task includes:

- a clear objective,
- an expected output,
- and dependencies on previous tasks where appropriate.

The agents collaborate sequentially, with each task building upon the output produced by the previous agent.

## Creating a Sample Environmental Monitoring Dataset

To demonstrate the CrewAI workflow without relying on external files, we generate a small synthetic environmental monitoring dataset.

The dataset represents observations exported from a remote sensing/GIS workflow and contains:

- NDVI (Normalized Difference Vegetation Index)
- Land Cover
- Rainfall
- Surface Temperature
- Vegetation Health

A few intentional data quality issues are introduced so that the Environmental Data Quality Specialist has meaningful work to perform before the analysis begins.

In [9]:
import pandas as pd

environmental_data = pd.DataFrame({
    "Plot_ID": [
        "P001","P002","P003","P004","P005",
        "P006","P007","P008","P009","P010",
        "P011","P012"
    ],

    "District": [
        "Faisalabad","Chiniot","Jhang","Toba Tek Singh",
        "Faisalabad","Chiniot","Jhang","Faisalabad",
        "Toba Tek Singh","Jhang","Chiniot","Faisalabad"
    ],

    "Latitude":[
        31.41,31.72,31.27,30.97,
        31.44,31.69,31.20,31.40,
        30.95,31.25,31.71,31.39
    ],

    "Longitude":[
        73.08,72.98,72.33,72.48,
        73.05,72.95,72.30,73.11,
        72.51,72.37,72.99,73.02
    ],

    "NDVI":[
        0.82,
        0.41,
        0.77,
        None,      # Missing value
        0.65,
        0.35,
        1.25,      # Invalid NDVI
        0.71,
        0.48,
        0.81,
        0.29,
        0.76
    ],

    "Land_Cover":[
        "Agriculture",
        "Barren",
        "Forest",
        "Agriculture",
        "Urban",
        "Barren",
        "Forest",
        "Agriculture",
        "Urban",
        "Forest",
        "Barren",
        "Agriculture"
    ],

    "Rainfall_mm":[
        102,
        45,
        118,
        93,
        None,      # Missing value
        38,
        122,
        110,
        65,
        119,
        40,
        108
    ],

    "Surface_Temp_C":[
        28.2,
        35.7,
        26.8,
        30.1,
        33.4,
        36.1,
        25.9,
        28.5,
        32.6,
        26.3,
        35.5,
        27.8
    ],

    "Vegetation_Health":[
        "Healthy",
        "Poor",
        "Excellent",
        "Healthy",
        "Moderate",
        "Poor",
        "Excellent",
        "Healthy",
        "Moderate",
        "Excellent",
        "Poor",
        "Healthy"
    ]
})

# Add one duplicate row intentionally
environmental_data = pd.concat(
    [environmental_data, environmental_data.iloc[[2]]],
    ignore_index=True
)

environmental_data.to_csv("environmental_monitoring_dataset.csv", index=False)

environmental_data

,Plot_ID,District,Latitude,Longitude,NDVI,Land_Cover,Rainfall_mm,Surface_Temp_C,Vegetation_Health
0,P001,Faisalabad,31.41,73.08,0.82,Agriculture,102.0,28.2,Healthy
1,P002,Chiniot,31.72,72.98,0.41,Barren,45.0,35.7,Poor
2,P003,Jhang,31.27,72.33,0.77,Forest,118.0,26.8,Excellent
3,P004,Toba Tek Singh,30.97,72.48,NaN,Agriculture,93.0,30.1,Healthy
4,P005,Faisalabad,31.44,73.05,0.65,Urban,NaN,33.4,Moderate
5,P006,Chiniot,31.69,72.95,0.35,Barren,38.0,36.1,Poor
6,P007,Jhang,31.20,72.30,1.25,Forest,122.0,25.9,Excellent
7,P008,Faisalabad,31.40,73.11,0.71,Agriculture,110.0,28.5,Healthy
8,P009,Toba Tek Singh,30.95,72.51,0.48,Urban,65.0,32.6,Moderate
9,P010,Jhang,31.25,72.37,0.81,Forest,119.0,26.3,Excellent


## Task Definitions

The workflow consists of three dependent tasks.

Each task consumes the output of the previous task, demonstrating collaborative problem solving within a CrewAI workflow.

In [10]:
# Task 1
from crewai import Task

quality_check_task = Task(
    description="""
Review the environmental_monitoring_dataset.csv file.

Use the available tool to inspect the dataset.

Identify:

- Missing values
- Duplicate records
- Invalid NDVI values
- Any other noticeable data quality issues

Summarize your findings.
""",

    expected_output="""
A structured markdown report including:

- Dataset Overview
- Missing Values
- Duplicate Records
- Invalid Environmental Measurements
- Recommendations before analysis
""",

    agent=environmental_data_quality_specialist
)

In [11]:
# Task 2
analysis_task = Task(
    description="""
Using the cleaned dataset information and the previous quality assessment,
analyze the environmental dataset.

Identify:

- vegetation health trends
- NDVI observations
- land cover distribution
- rainfall patterns

Highlight important environmental insights.
""",

    expected_output="""
A markdown report containing:

- Key Statistics
- Major Environmental Trends
- Areas of Concern
- Recommendations
""",

    context=[quality_check_task],

    agent=remote_sensing_analyst
)

In [12]:
# Task 3
report_task = Task(
    description="""
Prepare a professional environmental assessment report
for decision-makers using the previous analyses.

The report should be concise,
well-organized,
and suitable for stakeholders without technical expertise.
""",

    expected_output="""
A professional report with:

# Executive Summary

# Key Findings

# Environmental Risks

# Recommendations

# Conclusion
""",

    context=[analysis_task],

    agent=environmental_report_writer
)

## Assemble the Crew

The crew is configured using a **sequential process**, meaning each task executes in a predefined order.

The output produced by one agent becomes the context for the next agent, allowing the team to collaboratively solve the overall problem.

In [29]:
from crewai import Crew, Process

environmental_crew = Crew(
    agents=[
        environmental_data_quality_specialist,
        remote_sensing_analyst,
        environmental_report_writer
    ],

    tasks=[
        quality_check_task,
        analysis_task,
        report_task
    ],

    process=Process.sequential,

    verbose=False
)

In [34]:
result = environmental_crew.kickoff()

# Agent: Environmental Data Quality Specialist
## Task: 
Review the environmental_monitoring_dataset.csv file.

Use the available tool to inspect the dataset.

Identify:

- Missing values
- Duplicate records
- Invalid NDVI values
- Any other noticeable data quality issues

Summarize your findings.









 An unknown error occurred. Please check the details below.



RateLimitError: litellm.RateLimitError: RateLimitError: OpenAIException - litellm.RateLimitError: RateLimitError: OpenrouterException - {"error":{"message":"Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day","code":429,"metadata":{"headers":{"X-RateLimit-Limit":"50","X-RateLimit-Remaining":"0","X-RateLimit-Reset":"1784851200000"},"provider_name":null}},"user_id":"user_3GUvArjzh9FJa8mGIYlvBUelFoq"}. Received Model Group=coder
Available Model Group Fallbacks=['smart-lite', 'batch']
Error doing the fallback: litellm.RateLimitError: RateLimitError: CerebrasException - Tokens per day limit exceeded - too many tokens processed.. Received Model Group=batch
Available Model Group Fallbacks=['coder', 'smart-lite']
Error doing the fallback: litellm.RateLimitError: litellm.RateLimitError: geminiException - {
  "error": {
    "code": 429,
    "message": "You exceeded your current quota, please check your plan and billing details. For more informatio

In [ ]:
print(type(result))

In [21]:
import crewai
import rich

print("CrewAI:", crewai.__version__)
print("Rich:", rich.__version__)

CrewAI: 0.130.0


AttributeError: module 'rich' has no attribute '__version__'

In [22]:
import importlib.metadata

print(importlib.metadata.version("rich"))

14.3.4


## Prompt Refinement

During the initial execution, the output produced by the **Environmental Data Quality Specialist** was too narrative and lacked a consistent structure, making it difficult for the **Remote Sensing Analyst** to quickly extract key information.

To improve task handoff, the `expected_output` of the first task was updated to require a structured markdown report with clearly labeled sections (Dataset Overview, Missing Values, Duplicate Records, Invalid Environmental Measurements, and Recommendations).

This change produced a more predictable intermediate output, allowing the downstream agent to interpret and build upon the findings more effectively.